In [ ]:
from pathlib import Path
import numpy as np
import cv2
import imageio
from IPython.display import Video, display

# --- Input / output -----------------------------------------------------------
INPUT = Path("dataset/wall_pipeline/shaky.mp4")
OUT = INPUT.with_name(f"{INPUT.stem}_stabilized.mp4")

# LK = Lucas-Kanade optical flow (Stage 2). SIGMA = Gaussian smoothing on path
# parameters when building Q from R (Stage 5).
LK = dict(
    winSize=(21, 21),
    maxLevel=3,
    criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01),
)
SIGMA = 12.0


# --- Small utilities ----------------------------------------------------------


def to_3x3(M):
    if M is None:
        return np.eye(3)
    return np.vstack([M.astype(np.float64), [0, 0, 1]])


def gaussian_smooth1d(x, sigma):
    x = np.asarray(x, dtype=np.float64)
    r = max(1, int(3 * sigma))
    t = np.arange(-r, r + 1, dtype=np.float64)
    k = np.exp(-(t ** 2) / (2 * sigma ** 2))
    k /= k.sum()
    return np.convolve(np.pad(x, r, mode="edge"), k, mode="valid")


def similarity_Qs_from_Rs(Rs, sigma):
    tx = np.array([float(R[0, 2]) for R in Rs])
    ty = np.array([float(R[1, 2]) for R in Rs])
    th = np.unwrap(np.array([np.arctan2(float(R[1, 0]), float(R[0, 0])) for R in Rs]))
    sc = np.array([np.hypot(float(R[0, 0]), float(R[1, 0])) for R in Rs])
    log_sc = np.log(np.maximum(sc, 1e-9))
    sc_s = np.exp(gaussian_smooth1d(log_sc, sigma))
    tx_s = gaussian_smooth1d(tx, sigma)
    ty_s = gaussian_smooth1d(ty, sigma)
    th_s = gaussian_smooth1d(th, sigma)
    Qs = []
    for i in range(len(Rs)):
        c, s = np.cos(th_s[i]), np.sin(th_s[i])
        Qs.append(
            np.array(
                [
                    [sc_s[i] * c, -sc_s[i] * s, tx_s[i]],
                    [sc_s[i] * s, sc_s[i] * c, ty_s[i]],
                    [0.0, 0.0, 1.0],
                ],
                dtype=np.float64,
            )
        )
    return Qs


def crop_box(Ms, w, h):
    xmin, xmax, ymin, ymax = -np.inf, np.inf, -np.inf, np.inf
    corners = np.array([[0, 0, 1], [w, 0, 1], [w, h, 1], [0, h, 1]], dtype=np.float64).T
    for M in Ms:
        invM = np.linalg.inv(np.vstack([M.astype(np.float64), [0, 0, 1]]))
        d = invM @ corners
        xs, ys = d[0] / d[2], d[1] / d[2]
        xmin, xmax = max(xmin, xs.min()), min(xmax, xs.max())
        ymin, ymax = max(ymin, ys.min()), min(ymax, ys.max())
    inset = max(1, int(0.005 * min(w, h)))
    x1 = int(np.ceil(xmin)) + inset
    y1 = int(np.ceil(ymin)) + inset
    x2 = int(np.floor(xmax)) - inset
    y2 = int(np.floor(ymax)) - inset
    m = 0.05
    fx1, fy1, fx2, fy2 = int(w * m), int(h * m), int(w * (1 - m)), int(h * (1 - m))
    if x2 <= x1 or y2 <= y1 or (x2 - x1) * (y2 - y1) < 0.72 * w * h:
        return fx1, fy1, fx2, fy2
    return max(x1, fx1), max(y1, fy1), min(x2, fx2), min(y2, fy2)


# =============================================================================
# Pass 1 - Stage 1-3 (each frame pair): corners, LK track, RANSAC similarity
# Produces: list As of incremental 3x3 similarities (one per consecutive pair).
# =============================================================================
cap = cv2.VideoCapture(str(INPUT))
fps = cap.get(cv2.CAP_PROP_FPS) or 30
h, w = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)), int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
_, prev = cap.read()
prev_gray = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)
# Stage 1: Shi-Tomasi-style corners (goodFeaturesToTrack).
prev_pts = cv2.goodFeaturesToTrack(prev_gray, maxCorners=200, qualityLevel=0.01, minDistance=10)
As = []
while True:
    ret, curr = cap.read()
    if not ret:
        break
    gray = cv2.cvtColor(curr, cv2.COLOR_BGR2GRAY)
    affine = None
    if prev_pts is not None and len(prev_pts) > 0:
        # Stage 2: Lucas-Kanade sparse flow from previous corners to this frame.
        nxt, st, _ = cv2.calcOpticalFlowPyrLK(prev_gray, gray, prev_pts, None, **LK)
        if nxt is not None:
            a, b = prev_pts[st == 1], nxt[st == 1]
            if len(a) >= 2:
                # Stage 3: one similarity mapping prev->curr; RANSAC marks outlier tracks.
                affine, _ = cv2.estimateAffinePartial2D(a, b, method=cv2.RANSAC)
            prev_pts = b.reshape(-1, 1, 2).astype(np.float32)
            if len(prev_pts) < 50:
                prev_pts = cv2.goodFeaturesToTrack(gray, 200, 0.01, 10)
        else:
            prev_pts = cv2.goodFeaturesToTrack(gray, 200, 0.01, 10)
    else:
        prev_pts = cv2.goodFeaturesToTrack(gray, 200, 0.01, 10)
    As.append(to_3x3(affine))
    prev_gray = gray
cap.release()

# =============================================================================
# Stage 4: chain increments into cumulative raw camera path R.
# =============================================================================
Rs = [np.eye(3, dtype=np.float64)]
for A in As:
    Rs.append(Rs[-1] @ A)

# =============================================================================
# Stage 5: smooth R -> Q; warp M_i = Q_i @ inv(R_i); crop; second pass below.
# =============================================================================
Qs = similarity_Qs_from_Rs(Rs, SIGMA)
Ms = [(Qs[i] @ np.linalg.inv(Rs[i]))[:2].astype(np.float32) for i in range(1, len(Rs))]
x1, y1, x2, y2 = crop_box(Ms, w, h)

# =============================================================================
# Pass 2: read video again, apply each M, crop, write MP4, display in/out.
# =============================================================================
cap = cv2.VideoCapture(str(INPUT))
_, prev = cap.read()
frames = [cv2.cvtColor(prev[y1:y2, x1:x2], cv2.COLOR_BGR2RGB)]
for M in Ms:
    _, curr = cap.read()
    warped = cv2.warpAffine(curr, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    frames.append(cv2.cvtColor(warped[y1:y2, x1:x2], cv2.COLOR_BGR2RGB))
cap.release()

OUT.parent.mkdir(parents=True, exist_ok=True)
imageio.mimsave(
    str(OUT),
    frames,
    fps=fps,
    codec="libopenh264",
    macro_block_size=1,
    output_params=["-profile:v", "77", "-rc_mode", "off"],
)
display(Video(str(INPUT.resolve())))
display(Video(str(OUT.resolve())))

